In [ ]:
from openai import OpenAI
# Set OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

chat_response = client.chat.completions.create(
    model="Qwen/Qwen3-8B",
    messages=[
        {"role": "user", "content": "Give me a short introduction to large language models."},
    ],
    max_tokens=32768,
    temperature=0.6,
    top_p=0.95,
    extra_body={
        "top_k": 20,
    },
)
print("Chat response:", chat_response)

In [7]:
import argparse
import json
import requests
from pathlib import Path

API_BASE = "http://localhost:9000"


def upload_pdf(pdf_path: str):
    print(f"\nUploading: {pdf_path}")

    with open(pdf_path, "rb") as f:
        response = requests.post(
            f"{API_BASE}/upload",
            files={
                "file": (
                    Path(pdf_path).name,
                    f,
                    "application/pdf"
                )
            }
        )

    response.raise_for_status()

    result = response.json()

    print(
        f"Upload successful. "
        f"Indexed {result.get('chunks')} chunks."
    )

    return result


def ask_question(question: str):
    print(f"\nQuestion: {question}")

    response = requests.post(
        f"{API_BASE}/ask",
        json={
            "question": question
        }
    )

    response.raise_for_status()

    result = response.json()

    print("\nAnswer")
    print("=" * 80)
    print(result["answer"])

    print("\nRetrieved Chunks")
    print("=" * 80)

    for idx, chunk in enumerate(
        result.get("sources", []),
        start=1
    ):
        print(f"\nChunk {idx}")
        print("-" * 40)
        print(chunk[:500])

    return result


def interactive_mode():
    print("\nInteractive Question Mode")
    print("Type 'quit' to exit.\n")

    while True:
        question = input("> ").strip()

        if question.lower() in ["quit", "exit"]:
            break

        try:
            ask_question(question)

        except Exception as e:
            print(f"Error: {e}")


def main():
    parser = argparse.ArgumentParser(
        description="Test PDF RAG API"
    )

    parser.add_argument(
        "--pdf",
        required=True,
        help="Path to PDF"
    )

    parser.add_argument(
        "--question",
        help="Single question"
    )

    parser.add_argument(
        "--api",
        default="http://localhost:9000",
        help="FastAPI URL"
    )

    args = parser.parse_args()

    global API_BASE
    API_BASE = args.api

    upload_pdf(args.pdf)

    if args.question:
        ask_question(args.question)
    else:
        interactive_mode()


In [ ]:
args  = {
    "--pdf": 
}
